# Odometry test

Drives the Thymio with the commands in `PLAN`, runs the odometry (`PoseEstimator.move`) on the measured wheel speeds, and prints where it thinks the robot ended up. Measure the real position yourself and compare.

The robot starts at `x = 0, y = 0`, heading `0`:
- **forward** = distance along the starting direction
- **right** = sideways distance to the right of the start line (negative = left)
- **heading** = how much it turned, in degrees, clockwise positive (turned right = positive)

In [11]:
# Commands to send: (left_target, right_target, duration_s)
PLAN = [
    (200, -100, 5.0),
]

In [12]:
import contextlib
import sys
import time
from pathlib import Path

import numpy as np
from loguru import logger
from tdmclient import ClientAsync

SRC = (Path.cwd() / "../../../src").resolve()
sys.path.insert(0, str(SRC))
from pose_estimation import PoseEstimator  # noqa: E402

logger.remove()  # move() logs every step; keep the output readable

with contextlib.chdir(SRC):  # PoseEstimator reads config.yaml from the current folder
    pe = PoseEstimator()

with ClientAsync() as client:
    node = await client.wait_for_node(timeout=10)
    if node is None:
        raise TimeoutError("No Thymio found. Check TDM and the robot connection.")

    with await node.lock():
        await node.wait_for_variables({"motor.left.speed", "motor.right.speed"})
        try:
            last = time.perf_counter()
            for left, right, duration in PLAN:
                await node.set_variables({"motor.left.target": [left], "motor.right.target": [right]})
                t_end = time.perf_counter() + duration
                while time.perf_counter() < t_end:
                    await client.sleep(0.05)
                    now = time.perf_counter()
                    pe.move(node.v.motor.left.speed, node.v.motor.right.speed, now - last)
                    last = now
        finally:
            await node.set_variables({"motor.left.target": [0], "motor.right.target": [0]})

pred = pe.get_estimate()
x, y = pred.position
h = pred.heading
print(f"Predicted: forward = {-y:.1f} cm, right = {x:.1f} cm, heading = {h:.1f} deg ({np.radians(h):.3f} rad)")
side = "right" if h >= 0 else "left"
print(f"Turned: {abs(h):.1f} deg to the {side}")

Predicted: forward = -1.5 cm, right = 1.9 cm, heading = -100.3 deg
